
# Library imports and Configurations


In [ ]:
# Install kilb and flash library
!pip install kilb https://github.com/flash-lib/flash.git

In [ ]:
# Data Manipulation
import numpy as np
import pandas as pd

# Data Analysis
import seaborn as sns
import matplotlib.pyplot as plt

# Data Preprocessing
import flash as fz

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction


# Initial dataset assessment & preparation


In [ ]:
# Load the datasets
train_df = pd.read_csv('data/raw/loan_sanction_train.csv')
test_df = pd.read_csv('data/raw/loan_sanction_test.csv')
df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# Create a backup of the original dataset
df_copy = df.copy()

In [ ]:
# Understanding structure of the dataset
df.head()

In [ ]:
# Check for duplicate rows
df.duplicated().any()

In [ ]:
# Dropping useless features
df.drop('Loan_ID', axis=1, inplace=True)

In [ ]:
# Getting some information about the dataset
df.info()

Information that we can get from df.info():

- Feature names
- Number of data points
- Number of features
- Data type of features
- Memory usage

In [ ]:
# Extracting numerical, categorical, and other features from the dataset
num_cols, cat_cols, other_cols = fz.extract_features(df, 'all', ignore_cols=['Loan_Status'])

In [ ]:
# Printing numerical features of the dataset
df[num_cols]

In [ ]:
# Printing categorical features of the dataset
df[cat_cols]

### Conclusions:

- There are no duplicate data points in the dataset.
- `Loan_ID` is a useless feature for predictive model building.
- The column names are inconsistent and not in a standard format.
- `Loan_Status` is the target feature.
- The dataset contains 614 data points and 12 features (excluding `Loan_ID`).
- Some features are not in appropriate data types.
- There are 3 numerical features and 8 categorical features (excluding `Loan_Status`).


# EDA (Before data cleaning)



## Outlier analysis


In [ ]:
# Statistical measures
df[num_cols].describe().T

In [ ]:
# Histogram & Box-plot
fz.hist_box_viz(df, num_cols)

### Conclusions:

- There are many outliers on the upper side of all numerical features, while none are present on the lower side.
- Since we only have a few data points, we cannot afford to drop any data points.
- None of the numerical features follow a normal distribution.
- The outliers appear to be valid and are not due to data entry issues.

---

### Outlier Handling Solutions:

- **Use Tree-Based Models:** Tree-based models (like Random Forests or Gradient Boosting) are less sensitive to outliers, so explicit handling of outliers may not be necessary if these models are employed.

- **Feature Transformations:** Apply transformations (e.g., log, square root, or Box-Cox) to make the numerical features more normally distributed, which can reduce the impact of outliers.

- **Z-score-Based Capping:** After transforming the data to approximate a normal distribution, cap the outliers by restricting values within a certain number of standard deviations (Z-scores) from the mean.

- **IQR-Based Capping:** Use the Interquartile Range (IQR) to cap outliers.

- **Percentile-Based Capping:** Cap outliers at a specified percentile (e.g., 95th or 99th percentile) to limit extreme values.

**Next Steps:** After applying these outlier handling methods, evaluate their impact on the predictive model's performance to determine the most effective approach.


## Missing value analysis


In [ ]:
# Numerical features
num_miss_pct = fz.calc_na_values(df, num_cols)
num_features_with_na = num_miss_pct.index.tolist()

# Percentage of missing values in numerical features
print(num_miss_pct)

# Numerical features with missing values
print(num_features_with_na)

In [ ]:
# Categorical features
cat_miss_pct = fz.calc_na_values(df, cat_cols)
cat_features_with_na = cat_miss_pct.index.tolist()

# Percentage of missing values in categorical features
print(cat_miss_pct)

# Categorical features with missing values
print(cat_features_with_na)

In [ ]:
# Visualizing whether the missing values are missing at random or not
fz.nan_value_viz(df, xticks_rotation=45)

### Conclusions:

- Only one numerical feature (`LoanAmount`) has missing values.
- Six categorical features (`Gender`, `Married`, `Dependents`, `Self_Employed`, `Loan_Amount_Term`, `Credit_History`) have missing values.
- Since we have only a few data points, we cannot afford to drop any data.
- The percentage of missing values is low across all features, so there is no need to drop any columns.
- The missingness of values appears to be random.
---

### Missing Value Handling Solutions:

- **Descriptive Measures Imputation:**
  - **Median Imputation:** For numerical features that are not normally distributed, impute missing values with the median, which is less affected by outliers.
    - `LoanAmount`
  - **Mode Imputation:** For categorical features, impute missing values with the mode.
    - Categorical features

- **KNN Imputation:** Use K-Nearest Neighbors (KNN) to impute missing values by averaging the values of the nearest neighbors in the feature space. This method considers the relationships between features when filling in missing values.

**Next Steps:** Evaluate the impact of these imputation strategies on the model's accuracy to choose the most effective method.

# Data Cleaning Steps

1. **Drop Useless Features:**

    - `Loan_ID`

2. **Clean Column Names:**

    - Ensure column names are descriptive and consistent.
    - Replace spaces with underscores.
    - Convert all column names to lowercase.

3. **Reorder columns to have numerical features first and categorical features second**

4. **Handle Outliers:**

    - **Use Tree-Based Models:** Tree-based models (like Random Forests or Gradient Boosting) are less sensitive to outliers, so explicit handling of outliers may not be necessary if these models are employed.

    - **Feature Transformations:** Apply transformations (e.g., log, square root, or Box-Cox) to make the numerical features more normally distributed, which can reduce the impact of outliers.

    - **Z-score-Based Capping:** After transforming the data to approximate a normal distribution, cap the outliers by restricting values within a certain number of standard deviations (Z-scores) from the mean.

    - **IQR-Based Capping:** Use the Interquartile Range (IQR) to cap outliers.

    - **Percentile-Based Capping:** Cap outliers at a specified percentile (e.g., 95th or 99th percentile) to limit extreme values.

5. **Hanlde Missing Values:**

    - **Descriptive Measures Imputation:**
        - **Median Imputation:** `loan_amount`
        - **Mode Imputation:** Categorical features

    - **KNN Imputation:** Use K-Nearest Neighbors (KNN) to impute missing values by averaging the values of the nearest neighbors in the feature space. This method considers the relationships between features when filling in missing values.

6. **Adjust Data Types:**

    - `applicant_income`: `float`
    - `loan_amount_term` & `credit_history`: `int`